# Build your own view

Companion notebook for the RDDAC [documentation](https://rddac.readthedocs.io). It is written to stand on its own: every step is annotated so the notebook reads top to bottom.

## Walkthrough

1. load the dataset and list the published RecordSets,
2. look up the available source fields in the `field-map` RecordSet,
3. append a custom RecordSet with slicing,
4. mix in `process-parameters` CSV columns,
5. inspect the resolved source fields and transforms,
6. stream records of the view with `rddac.streaming.iter_view`,
7. register the prefab recipes from `rddac.views`,
8. read the experiment index directly.

## Assumptions

- This notebook lives in `notebooks/` of the repository and reads the dataset from `../data/` (on your machine: the directory `rddac download` wrote to).
- The data directory contains `metadata.json`, `process_parameters.csv`, and at least the bundled `sample.zip`.

In [1]:
import warnings
warnings.filterwarnings('ignore')

import rddac
print('rddac', rddac.__version__)

from pathlib import Path
DATA_DIR = Path('../data')
# DATA_DIR = Path('./data')   # uncomment instead when running from the repository root
assert (DATA_DIR / 'metadata.json').is_file(), f'{DATA_DIR} does not contain the dataset'

rddac 1.1.0


## 1. Load the dataset

`rddac.load(data_dir=...)` reads `metadata.json` from `data_dir`, registers any zip files it finds there so the HDF5 members can be read in place, and exposes the published RecordSets through `ds.metadata.record_sets`. Listing them first is good practice: a custom view is only ever needed when none of the published ones fit.

Four h5-backed views ship with the manifest — `force-curve`, `thickness`, `pointcloud-op10`, and `pointcloud-op20` — plus the `process-parameters` index and the `field-map` lookup table that custom views build on.

In [2]:
ds = rddac.load(data_dir=DATA_DIR)
print('published RecordSets:')
for rs in ds.metadata.record_sets:
    print(f'  {rs.id}')

published RecordSets:
  process-parameters
  field-map
  pointcloud-op10
  pointcloud-op20
  force-curve
  sheet-thickness
  oil-thickness
  thickness


## 2. The `field-map` RecordSet

Every HDF5 dataset that a view can pull from is declared once in the manifest's `field-map` RecordSet: the field name is what goes on the right-hand side of `add_view(fields=...)`, and the description documents what the array contains. The helper below prints the full lookup table, so there is no need to memorise anything.

In [3]:
from rddac.croissant import field_map

for name, f in field_map(ds).items():
    print(f'  {name:32s} {(f.description or "")}')

  force_data                       Press force and process signals, time series. Columns: [time (s), load_cell_1..4 (kN), punch_temp (degC), punch_pos (mm), total_force (kN)]. HDF5 path: force/data. Shape (per file): (1140, 8).
  oil_thickness_data               Lubricant film measurement along a sensor traverse. Columns: [sensor_position (mm), oil_value (g/m^2)]. HDF5 path: oil_thickness/data. Shape (per file): (421, 2).
  pointcloud_op10_luminescence     Flattened luminescence/intensity buffer of the OP10 scan, pixel-aligned with op10/z. HDF5 path: pointcloud/op10/luminescence. Shape (per file): (6400000).
  pointcloud_op10_z                Flattened height (Z) buffer of the OP10 laser scan (after deep drawing), row-major over a y_shape x x_shape grid (see group attributes). Reshape and apply per-pixel calibration to recover a 3D surface (see the rddac preprocessing step). HDF5 path: pointcloud/op10/z. Shape (per file): (6400000).
  pointcloud_op20_luminescence     Flattened luminesc

## 3. Append a custom RecordSet

`rddac.add_view(ds, name, fields)` mutates `ds` in place. After it returns, the new RecordSet is part of `ds.metadata.record_sets` and can be streamed exactly like a published one.

Each entry of `fields` maps an **alias** (the dict key) to a **source field** from `field-map`, plus an optional **index selection** along the first axis. The raw `force/data` table has shape `(n, 8)` with `n` varying per experiment, which makes it a good demonstrator for every supported shape:

- `force` keeps the whole table -> `(n, 8)`, the form for a model that consumes the full press stroke.
- `force_head` takes rows `0..99` -> `(100, 8)`, a fixed-size window (handy for batching, see the PyTorch notebook).
- `first_sample` takes row `0` -> `(8,)`, a single snapshot; the integer index drops the axis.
- `sheet` is the shortcut form: a bare string means "the whole field", same as `("...", None)`.

The published manifest on DaRUS is **not** modified; only the in-memory dataset grows.

In [4]:
rddac.add_view(
    ds,
    'my-view',
    fields={
        'force':        ('force_data', None),             # whole (n, 8) table
        'force_head':   ('force_data', list(range(100))), # fixed 100-row window
        'first_sample': ('force_data', 0),                # one row
        'sheet':        'sheet_thickness_data',           # whole field, shortcut form
    },
)

print('record_sets after add_view:')
for rs in ds.metadata.record_sets:
    print(f'  {rs.id}')

record_sets after add_view:
  process-parameters
  field-map
  pointcloud-op10
  pointcloud-op20
  force-curve
  sheet-thickness
  oil-thickness
  thickness
  my-view


## 4. Mix in process-parameters columns

`add_view` also accepts **qualified** field IDs of the form `"<record-set>/<field>"`, which lets a single view combine HDF5 fields from `field-map` with CSV columns from `process-parameters`. The two sources are joined on the experiment id automatically: `mlcroissant` requires an explicit cross-source `references` link to validate the manifest, and `add_view` injects one on the first field-map field for you, so no manual join wiring is needed on the consumer side.

Iteration via `rddac.streaming.iter_view` returns one record per experiment with both kinds of fields side by side: HDF5 arrays and CSV columns appear under their respective aliases. (At this layer the CSV goes through pandas, so strings arrive as `str`, not `bytes`.)

In [5]:
rddac.add_view(
    ds,
    'my-view-with-params',
    fields={
        'force_head':         ('force_data', list(range(100))),
        'geometry':           'process-parameters/geometry',
        'blankholder_force':  'process-parameters/blankholder_force',
        'oil_type':           'process-parameters/oil_type',
        'split':              'process-parameters/split',
    },
)

for rec in rddac.streaming.iter_view(
    'my-view-with-params',
    data_dir=DATA_DIR,
    dataset=ds,
    sim_ids=[0, 4500],
):
    print(f"experiment {rec['_sim_id']:4d}: force_head.shape={rec['force_head'].shape}  "
          f"geometry={rec['geometry']!r}  blankholder_force={rec['blankholder_force']} kN  "
          f"oil_type={rec['oil_type']!r}  split={rec['split']!r}")

experiment    0: force_head.shape=(100, 8)  geometry='concave'  blankholder_force=100 kN  oil_type='coarse'  split='val'
experiment 4500: force_head.shape=(100, 8)  geometry='convex'  blankholder_force=100 kN  oil_type='coarse'  split='train'


## 5. Inspect the new view's fields

Each field of the view stores a **source** (the field-map entry it pulls from) and an **optional JSONPath transform** that records the index selection:

- `("field_id", 0)` -> JSONPath `$[0]`
- `("field_id", [0, 1, ...])` -> JSONPath `$[0,1,...]`
- `"field_id"` or `("field_id", None)` -> no transform (whole field)

Printing the resolved transforms is a quick way to verify the view was built as expected before you start training on it.

In [6]:
view = next(rs for rs in ds.metadata.record_sets if rs.id == 'my-view')
for f in view.fields:
    transforms = [t.json_path for t in (f.source.transforms or [])]
    shown = [t if len(t) <= 30 else t[:27] + '...' for t in transforms]
    print(f'  {f.id:24s} <- {f.source.uuid:28s} transforms={shown}')

  my-view/force            <- field-map/force_data         transforms=[]
  my-view/force_head       <- field-map/force_data         transforms=['$[0,1,2,3,4,5,6,7,8,9,10,11...']
  my-view/first_sample     <- field-map/force_data         transforms=['$[0]']
  my-view/sheet            <- field-map/sheet_thickness_data transforms=[]


## 6. Stream records of the view

> **Heads up.** Do **not** iterate an h5-backed view through `ds.records(view)`. `mlcroissant` walks the whole FileSet at iterator setup and materialises every HDF5 member of every referenced zip before applying any filter — with the full release local (two ~40 GB zips) it did not produce a first record within several minutes in our tests. This path is fine for CSV-backed RecordSets (step 8 below) but the wrong tool for the measurement data.
>
> Two replacements both work and both scale to the full release. They share the same lazy `experiment id -> local zip` index, open a zip only when a record is actually requested, and silently skip experiments whose zip is missing on a partial download. Both accept the `dataset=` kwarg so the in-memory `add_view` mutation flows through:
>
> ```python
> # No-PyTorch path: plain Python generator (this notebook + 06_streaming).
> for rec in rddac.streaming.iter_view('my-view', data_dir=DATA_DIR, dataset=ds):
>     ...
>
> # PyTorch path: torch.utils.data.IterableDataset for DataLoader batching (03_pytorch).
> from rddac import RDDACDataset
> custom_ds = RDDACDataset(view='my-view', data_dir=DATA_DIR, dataset=ds)
> ```

Each record is a dict keyed by the aliases declared in `add_view`, with values already sliced according to the JSONPath transform, plus a private `_sim_id` key carrying the experiment id. `first_sample` comes out as `(8,)` (the `$[0]` transform dropped the first axis) while `force` keeps the full per-experiment `(n, 8)` shape — note how `n` differs between the two experiments below.

In [7]:
import numpy as np

for rec in rddac.streaming.iter_view('my-view', data_dir=DATA_DIR, dataset=ds, sim_ids=[0, 4500]):
    print(f"experiment {rec['_sim_id']:4d}")
    for k, v in rec.items():
        if k.startswith('_'):
            continue
        arr = np.asarray(v)
        print(f'  {k:14s} shape={arr.shape}  dtype={arr.dtype}')

experiment    0
  force          shape=(1140, 8)  dtype=float32
  force_head     shape=(100, 8)  dtype=float32
  first_sample   shape=(8,)  dtype=float32
  sheet          shape=(208, 2)  dtype=float32
experiment 4500
  force          shape=(720, 8)  dtype=float32
  force_head     shape=(100, 8)  dtype=float32
  first_sample   shape=(8,)  dtype=float32
  sheet          shape=(208, 2)  dtype=float32


## 8. Read the experiment index

The `process-parameters` RecordSet is sourced from the CSV (not from the h5 zips), so the plain `mlcroissant` records iterator works regardless of which zips are downloaded locally — this is the one RecordSet where `ds.records(...)` is the canonical access path. For tabular work, `pandas.read_csv(DATA_DIR / 'process_parameters.csv')` is one line away.

In [8]:
for n, rec in enumerate(ds.records('process-parameters'), start=1):
    if n == 1:
        for k, v in rec.items():
            print(f'  {k:42s} = {v}')
    if n >= 1:
        break

  process-parameters/index                   = 0
  process-parameters/experiment_id           = 1
  process-parameters/category                = 0
  process-parameters/geometry                = b'concave'
  process-parameters/blankholder_force       = 100
  process-parameters/mean_punch_temp         = 20.2
  process-parameters/oil_type                = b'coarse'
  process-parameters/has_pointcloud          = True
  process-parameters/has_oil                 = True
  process-parameters/split                   = b'val'
